In [ ]:
# 1. 导入必要的包
import os
import sys
import re
import importlib
import pandas as pd
from datasets import load_from_disk

# 清空缓存的模块
if 'dataset' in sys.modules:
    del sys.modules['dataset']

sys.path.insert(0, '/home/y-guo/self-ensemble/GYB_self-ensemble')
from dataset import LogiQAParaphraseDataset

print("✅ 导入成功")

ImportError: cannot import name 'LogiQAParaphraseDataset' from 'dataset' (/home/y-guo/self-ensemble/GYB_self-ensemble/dataset.py)

In [ ]:
# 2. 创建 dataset 实例
print("创建 LogiQAParaphraseDataset...")
dataset = LogiQAParaphraseDataset(model_name="llama3.2_3b_it")
print(f"✅ Dataset 创建成功")
print(f"Dataset 大小: {len(dataset.ds)}")
print(f"Dataset root: {dataset.dataset_root}")

创建 CommonsenseParaphraseDataset...
Dataset already exists at /home/y-guo/self-ensemble/commonsense_paraphrase/llama3.2_3b_it/paraphrases_dataset. Loading from disk.
✅ Dataset 创建成功
Dataset 大小: 50
Dataset root: /home/y-guo/self-ensemble/commonsense_paraphrase/llama3.2_3b_it


In [4]:
# 3. 检查 instruction（多选问题格式）
print("Multi-Choice Instruction:")
print("=" * 70)
print(dataset.instruction)
print("=" * 70)

Multi-Choice Instruction:
Multiple-Choice Question Answering
Your task is to select the correct answer to the question from the given options.
Consider only the provided options and choose the single most appropriate one.

Output format constraint:
• Output exactly one capital letter corresponding to the chosen option
• Do not output punctuation, text, or explanations


In [5]:
# 4. 查看第一条数据
print("第一条数据结构:")
first_item = dataset.ds[0]
for key in first_item.keys():
    value = first_item[key]
    if isinstance(value, list) and len(value) > 3:
        print(f"  {key}: (list with {len(value)} items)")
        print(f"    First 2: {value[:2]}")
    else:
        print(f"  {key}: {value}")

第一条数据结构:
  uuid: 02e821a3e53cb320790950aab4489e85
  paraphrases: (list with 11 items)
    First 2: ['What have driving navigation apps like Google Maps largely taken the place of?', 'Which traditional tool have digital road and highway mapping services mostly supplanted?']
  answers: ['atlas']
  answer_label: D
  choices_label: (list with 5 items)
    First 2: ['A', 'B']
  choices_text: (list with 5 items)
    First 2: ['united states', 'mexico']
  orig_question: Google Maps and other highway and street GPS services have replaced what?
  question_concept: highway


In [6]:
# 5. 测试 collate_fn（应该返回 6 个值）
print("测试 collate_fn...")
batch = [dataset.ds[i] for i in range(2)]
result = dataset.collate_fn(batch)

print(f"\nCollate_fn 返回值数量: {len(result)}")
print(f"期望: 6 (uuids, answers, paraphrases, choices_label, choices_text, answer_label)")

if len(result) == 6:
    uuids, answers, all_paraphrases, choices_labels, choices_texts, answer_labels = result
    print(f"\n✅ 成功解包 6 个值")
    print(f"  uuids: {uuids}")
    print(f"  answers: {answers}")
    print(f"  paraphrases 数量: {len(all_paraphrases)} versions")
    print(f"  choices_labels[0]: {choices_labels[0]}")
    print(f"  choices_texts[0]: {choices_texts[0]}")
    print(f"  answer_labels: {answer_labels}")
else:
    print(f"\n❌ 返回值数量不正确: {len(result)}")

测试 collate_fn...

Collate_fn 返回值数量: 6
期望: 6 (uuids, answers, paraphrases, choices_label, choices_text, answer_label)

✅ 成功解包 6 个值
  uuids: ['02e821a3e53cb320790950aab4489e85', '0476192858e7f541611b5c2d3c5e5197']
  answers: [['atlas'], ['being found out']]
  paraphrases 数量: 11 versions
  choices_labels[0]: ['A', 'B', 'C', 'D', 'E']
  choices_texts[0]: ['united states', 'mexico', 'countryside', 'atlas', 'oceans']
  answer_labels: ['D', 'C']


In [7]:
# 6. 测试 format_example（few-shot 格式）
print("测试 format_example (few-shot 格式):")
print("=" * 70)
example = dataset.ds[0]
formatted = dataset.format_example(example)
print(formatted)
print("=" * 70)

测试 format_example (few-shot 格式):
Question:
What have driving navigation apps like Google Maps largely taken the place of?

Options:
A. united states
B. mexico
C. countryside
D. atlas
E. oceans

Answer (A–E only): D


In [8]:
# 7. 测试 get_few_shot_examples
print("生成 few-shot examples (k=2)...")
few_shot_examples = dataset.get_few_shot_examples(k=2, seed=42)
print("=" * 70)
print(few_shot_examples)
print("=" * 70)

生成 few-shot examples (k=2)...
Question:
When the boss told his subordinate to handle errands, was that a signal to visit the suppliers to collect payment from them?

Options:
A. park
B. make time for
C. receive instructions
D. take money
E. leave work

Answer (A–E only): D

Question:
Where did he discover the moldy forgotten food stored in the rear of what piece of furniture or appliance?

Options:
A. carpet
B. refrigerator
C. breadbox
D. fridge
E. coach

Answer (A–E only): B


In [9]:
# 8. 导入并测试 construct_multi_choice_prompts
from parallel_ensemble import construct_multi_choice_prompts

print("测试 construct_multi_choice_prompts 函数...")

# 使用第一条数据
test_item = dataset.ds[0]
test_question = test_item["paraphrases"][0]
test_choices_label = test_item["choices_label"]
test_choices_text = test_item["choices_text"]

print(f"\n测试问题: {test_question[:80]}...")
print(f"选项标签: {test_choices_label}")
print(f"正确答案: {test_item['answer_label']}")

测试 construct_multi_choice_prompts 函数...

测试问题: What have driving navigation apps like Google Maps largely taken the place of?...
选项标签: ['A', 'B', 'C', 'D', 'E']
正确答案: D


In [10]:
# 9. 构造不带 few-shot 的 prompt
print("\n不带 few-shot examples 的 prompt:")
print("=" * 70)
prompt_no_fewshot = construct_multi_choice_prompts(
    dataset.instruction,
    "",  # 无 few-shot
    test_question,
    test_choices_label,
    test_choices_text
)
print(prompt_no_fewshot[0])
print("=" * 70)


不带 few-shot examples 的 prompt:
Multiple-Choice Question Answering
Your task is to select the correct answer to the question from the given options.
Consider only the provided options and choose the single most appropriate one.

Output format constraint:
• Output exactly one capital letter corresponding to the chosen option
• Do not output punctuation, text, or explanations

Question:
What have driving navigation apps like Google Maps largely taken the place of?

Options:
A. united states
B. mexico
C. countryside
D. atlas
E. oceans

Answer (A–E only):


In [11]:
# 10. 构造带 few-shot 的 prompt
print("\n带 few-shot examples 的 prompt:")
print("=" * 70)
few_shot = dataset.get_few_shot_examples(k=1, seed=42)
prompt_with_fewshot = construct_multi_choice_prompts(
    dataset.instruction,
    few_shot,
    test_question,
    test_choices_label,
    test_choices_text
)
print(prompt_with_fewshot[0])
print("=" * 70)


带 few-shot examples 的 prompt:
Multiple-Choice Question Answering
Your task is to select the correct answer to the question from the given options.
Consider only the provided options and choose the single most appropriate one.

Output format constraint:
• Output exactly one capital letter corresponding to the chosen option
• Do not output punctuation, text, or explanations

Question:
When the boss told his subordinate to handle errands, was that a signal to visit the suppliers to collect payment from them?

Options:
A. park
B. make time for
C. receive instructions
D. take money
E. leave work

Answer (A–E only): D

Question:
What have driving navigation apps like Google Maps largely taken the place of?

Options:
A. united states
B. mexico
C. countryside
D. atlas
E. oceans

Answer (A–E only):


In [12]:
# 11. 测试答案提取逻辑（正则表达式）
print("\n测试答案提取逻辑:")
print("=" * 70)

test_generations = [
    "A",
    "B.",
    "The answer is C",
    "I think the correct answer is D because...",
    "E. This is the best option",
    "Based on the context, A is correct",
    "No valid answer here",
    "F is not valid",
]

for gen in test_generations:
    match = re.search(r'[A-E]', gen)
    prediction = match.group(0) if match else ""
    print(f"  生成: '{gen:40s}' -> 提取: '{prediction}'")

print("=" * 70)


测试答案提取逻辑:
  生成: 'A                                       ' -> 提取: 'A'
  生成: 'B.                                      ' -> 提取: 'B'
  生成: 'The answer is C                         ' -> 提取: 'C'
  生成: 'I think the correct answer is D because...' -> 提取: 'D'
  生成: 'E. This is the best option              ' -> 提取: 'E'
  生成: 'Based on the context, A is correct      ' -> 提取: 'B'
  生成: 'No valid answer here                    ' -> 提取: ''
  生成: 'F is not valid                          ' -> 提取: ''


In [13]:
# 12. 测试 DataLoader 完整流程
print("\n测试 DataLoader 完整流程...")
dataloader = dataset.get_dataloader(batch_size=2, shuffle=False)

print("获取第一个 batch...")
for batch_data in dataloader:
    print(f"\nBatch 数据包含 {len(batch_data)} 个元素")
    
    # 解包
    uuids, answers, all_paraphrases, choices_labels, choices_texts, answer_labels = batch_data
    
    print(f"\nBatch size: {len(uuids)}")
    print(f"UUIDs: {uuids}")
    print(f"Answer labels: {answer_labels}")
    print(f"Paraphrase versions: {len(all_paraphrases)}")
    
    print(f"\n第一个样本:")
    print(f"  UUID: {uuids[0]}")
    print(f"  Answer: {answers[0][0]}")
    print(f"  Answer label: {answer_labels[0]}")
    print(f"  Choices: {list(zip(choices_labels[0], choices_texts[0]))}")
    print(f"  第一个 paraphrase: {all_paraphrases[0][0][:80]}...")
    
    break

print("\n✅ 所有测试通过！")


测试 DataLoader 完整流程...
获取第一个 batch...

Batch 数据包含 6 个元素

Batch size: 2
UUIDs: ['02e821a3e53cb320790950aab4489e85', '0476192858e7f541611b5c2d3c5e5197']
Answer labels: ['D', 'C']
Paraphrase versions: 11

第一个样本:
  UUID: 02e821a3e53cb320790950aab4489e85
  Answer: atlas
  Answer label: D
  Choices: [('A', 'united states'), ('B', 'mexico'), ('C', 'countryside'), ('D', 'atlas'), ('E', 'oceans')]
  第一个 paraphrase: What have driving navigation apps like Google Maps largely taken the place of?...

✅ 所有测试通过！


In [14]:
# 13. 模拟 parallel_ensemble.py 中的处理流程
print("\n模拟 parallel_ensemble.py 处理流程:")
print("=" * 70)

flag_multi_choice = True
dataloader = dataset.get_dataloader(batch_size=1, shuffle=False)

for batch_data in dataloader:
    if flag_multi_choice:
        uuids, answers, all_paraphrases, choices_labels, choices_texts, answer_labels = batch_data
        print("✅ 成功解包 6 个值 (multi-choice mode)")
    else:
        uuids, answers, all_paraphrases = batch_data
        print("✅ 成功解包 3 个值 (non-multi-choice mode)")
    
    print(f"\n处理样本: {uuids[0]}")
    print(f"Answer label: {answer_labels[0]}")
    
    # 构造 prompt (使用第一个 paraphrase)
    para = all_paraphrases[0][0]
    
    if flag_multi_choice:
        prompt = construct_multi_choice_prompts(
            dataset.instruction,
            "",  # 无 few-shot
            para,
            choices_labels[0],
            choices_texts[0]
        )
        print(f"\n✅ 使用 construct_multi_choice_prompts")
    else:
        prompt = dataset.construct_prompts("", [para])
        print(f"\n✅ 使用 dataset.construct_prompts")
    
    print(f"\nPrompt 长度: {len(prompt[0])} 字符")
    print(f"Prompt 预览 (前 200 字符):")
    print(prompt[0][:200] + "...")
    
    # 模拟生成和答案提取
    mock_generation = "C. This is the correct answer."
    if flag_multi_choice:
        match = re.search(r'[A-E]', mock_generation)
        prediction = match.group(0) if match else ""
        print(f"\n模拟生成: '{mock_generation}'")
        print(f"提取答案: '{prediction}'")
    else:
        prediction = mock_generation.strip().split()[0] if mock_generation.strip() else ""
        print(f"\n非多选提取: '{prediction}'")
    
    break

print("\n" + "=" * 70)
print("✅ 完整流程测试通过！")


模拟 parallel_ensemble.py 处理流程:
✅ 成功解包 6 个值 (multi-choice mode)

处理样本: 02e821a3e53cb320790950aab4489e85
Answer label: D

✅ 使用 construct_multi_choice_prompts

Prompt 长度: 525 字符
Prompt 预览 (前 200 字符):
Multiple-Choice Question Answering
Your task is to select the correct answer to the question from the given options.
Consider only the provided options and choose the single most appropriate one.

Out...

模拟生成: 'C. This is the correct answer.'
提取答案: 'C'

✅ 完整流程测试通过！


In [15]:
# 14. 加载模型和tokenizer（只需运行一次）
from transformers import AutoModelForCausalLM, AutoTokenizer
from constants import MODEL_PATHs

model_name = "llama3.2_3b_it"
model_path = MODEL_PATHs.get(model_name)

print(f"加载模型: {model_path}")
print("这可能需要几分钟...")

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

print("✅ 模型加载完成！")
print(f"模型设备: {model.device}")

加载模型: meta-llama/Llama-3.2-3B-Instruct
这可能需要几分钟...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 模型加载完成！
模型设备: cuda:1


In [16]:
# 15. 创建迭代器（只需运行一次）
from parallel_ensemble import ensemble_generation, sample_paraphrases_per_item

# 创建 dataloader 迭代器
dataloader = dataset.get_dataloader(batch_size=8, shuffle=False)
dataloader_iter = iter(dataloader)

# Few-shot examples
few_shot_examples = dataset.get_few_shot_examples(k=5, seed=42)

# 准备所有样本
all_samples_queue = []
print("准备样本队列...")

for batch_data in dataloader:
    uuids, answers, all_paraphrases, choices_labels, choices_texts, answer_labels = batch_data
    
    # Sample paraphrases
    samples = sample_paraphrases_per_item(
        uuids=uuids,
        all_paraphrases=all_paraphrases,
        num_paraphrases=5,
        num_samples=1,  # 每个问题只生成1个sample用于快速测试
        repeat_paras=False
    )
    
    for uuid, sampled_paraphrases in samples:
        idx = uuids.index(uuid)
        all_samples_queue.append({
            'uuid': uuid,
            'answer': answers[idx],
            'answer_label': answer_labels[idx],
            'paraphrases': sampled_paraphrases,
            'choices_label': choices_labels[idx],
            'choices_text': choices_texts[idx]
        })

print(f"✅ 准备完成！共 {len(all_samples_queue)} 个样本待生成")
sample_idx = 0  # 当前样本索引

准备样本队列...
✅ 准备完成！共 50 个样本待生成


In [22]:
all_paraphrases

[('After spending a long time on his hobby, Johnny took a seat on a bench and unwound — where might this be located?',
  'What kind of discomfort can result from sitting very near the television for a long time?'),
 ('Johnny relaxed on a bench after extensive hobby activity; in what kind of place is he sitting?',
  'If someone watches television from a very short distance, what type of pain might they experience?'),
 ('Having finished a lot of work on his pastime, Johnny sat down on a bench to rest — where could this setting be?',
  'Watching TV up close for extended periods can lead to what kind of physical pain?'),
 ('Johnny cooled off on a bench after working hard on his hobby; what sort of location is he likely in?',
  'What sort of ache can be caused by sitting too close to the television while viewing?'),
 ('After doing heavy work on his hobby, Johnny sat on a bench and relaxed — what type of spot is this?',
  'Sitting close to the screen while watching TV could induce which kind

In [25]:
# 16. 🔄 运行此cell生成下一个样本（可重复运行）
if sample_idx >= len(all_samples_queue):
    print("⚠️ 所有样本已生成完毕！")
    print(f"重置索引以重新开始...")
    sample_idx = 0

# 获取当前样本
sample = all_samples_queue[sample_idx]
print(f"\n{'='*70}")
print(f"📝 正在生成样本 {sample_idx + 1}/{len(all_samples_queue)}")
print(f"{'='*70}")
print(f"UUID: {sample['uuid']}")
print(f"正确答案: {sample['answer_label']} - {sample['answer'][0]}")
print(f"使用 {len(sample['paraphrases'])} 个 paraphrases")

# 构造 prompts
all_prompts = []
for para in sample['paraphrases']:
    prompt = construct_multi_choice_prompts(
        dataset.instruction,
        few_shot_examples,
        para,
        sample['choices_label'],
        sample['choices_text']
    )
    all_prompts.append(prompt)
    print(prompt)
print(f"\n🚀 开始 ensemble generation...")

# Ensemble generation
generation = ensemble_generation(
    model,
    tokenizer,
    prompt_sets=all_prompts,
    integration_method="avg",  # 使用 logits averaging
    weights=None,
    max_new_tokens=10,
    ensemble_method=None,
    multilayer=False,
    token_mode="last",
    ensemble_layer_idx=15,
    ensemble_alpha=1.0
)

# 提取答案
match = re.search(r'[A-E]', generation.strip())
prediction = match.group(0) if match else ""

# 显示结果
print(f"\n{'='*70}")
print(f"📊 生成结果:")
print(f"{'='*70}")
print(f"生成文本: '{generation}'")
print(f"提取答案: '{prediction}'")
print(f"正确答案: '{sample['answer_label']}'")
print(f"判断结果: {'✅ 正确' if prediction == sample['answer_label'] else '❌ 错误'}")

# 显示选项
print(f"\n选项:")
for label, text in zip(sample['choices_label'], sample['choices_text']):
    marker = "✓" if label == sample['answer_label'] else " "
    pred_marker = "→" if label == prediction else " "
    print(f"  {marker} {pred_marker} {label}. {text}")

print(f"\n{'='*70}\n")

# 移动到下一个样本
sample_idx += 1


📝 正在生成样本 8/50
UUID: 3d0f8824ea83ddcc9ab03055658b89d3
正确答案: B - refrigerator
使用 5 个 paraphrases
['Multiple-Choice Question Answering\nYour task is to select the correct answer to the question from the given options.\nConsider only the provided options and choose the single most appropriate one.\n\nOutput format constraint:\n• Output exactly one capital letter corresponding to the chosen option\n• Do not output punctuation, text, or explanations\n\nQuestion:\nWhen the boss told his subordinate to handle errands, was that a signal to visit the suppliers to collect payment from them?\n\nOptions:\nA. park\nB. make time for\nC. receive instructions\nD. take money\nE. leave work\n\nAnswer (A–E only): D\n\nQuestion:\nWhere did he discover the moldy forgotten food stored in the rear of what piece of furniture or appliance?\n\nOptions:\nA. carpet\nB. refrigerator\nC. breadbox\nD. fridge\nE. coach\n\nAnswer (A–E only): B\n\nQuestion:\nSean had lied about the corpse and felt terrified; what did h